In [1]:
import pandas as pd

file_path = "/home/phelix/Desktop/Projects/netflix-content-analysis/data/raw/Dataset.csv"
df = pd.read_csv(file_path)

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB
None


In [2]:
print(df.head())

  show_id     type                             title         director  \
0      s1    Movie              Dick Johnson Is Dead  Kirsten Johnson   
1      s3  TV Show                         Ganglands  Julien Leclercq   
2      s6  TV Show                     Midnight Mass    Mike Flanagan   
3     s14    Movie  Confessions of an Invisible Girl    Bruno Garotti   
4      s8    Movie                           Sankofa     Haile Gerima   

         country date_added  release_year rating  duration  \
0  United States  9/25/2021          2020  PG-13    90 min   
1         France  9/24/2021          2021  TV-MA  1 Season   
2  United States  9/24/2021          2021  TV-MA  1 Season   
3         Brazil  9/22/2021          2021  TV-PG    91 min   
4  United States  9/24/2021          1993  TV-MA   125 min   

                                           listed_in  
0                                      Documentaries  
1  Crime TV Shows, International TV Shows, TV Act...  
2                 TV Dr

In [3]:
df = df.replace(r'^\s*$', pd.NA, regex=True)

df["director"] = df["director"].fillna("Unknown Director")
df["country"] = df["country"].fillna("Unknown Country")
df["rating"] = df["rating"].fillna("Not Rated")
df["duration"] = df["duration"].fillna("Unknown Duration")
df["listed_in"] = df["listed_in"].fillna("Uncategorized")

df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

df = df.dropna(subset=["title", "type"])

In [4]:
df = df.drop_duplicates()
print("Rows after dropping exact duplicates:", len(df))

Rows after dropping exact duplicates: 8790


In [5]:
duplicate_check = df.duplicated(subset=["title", "country", "release_year"], keep=False)
print("Potential near-duplicates records:", duplicate_check.sum())

Potential near-duplicates records: 6


In [6]:
text_columns = ["type", "title", "director", "country", "rating", "duration", "listed_in"]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)

In [7]:
# quality check
print(df.isna().sum())
print(df.duplicated().sum())
print(df.head())

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64
0
  show_id     type                             title         director  \
0      s1    Movie              Dick Johnson Is Dead  Kirsten Johnson   
1      s3  TV Show                         Ganglands  Julien Leclercq   
2      s6  TV Show                     Midnight Mass    Mike Flanagan   
3     s14    Movie  Confessions of an Invisible Girl    Bruno Garotti   
4      s8    Movie                           Sankofa     Haile Gerima   

         country date_added  release_year rating  duration  \
0  United States 2021-09-25          2020  PG-13    90 min   
1         France 2021-09-24          2021  TV-MA  1 Season   
2  United States 2021-09-24          2021  TV-MA  1 Season   
3         Brazil 2021-09-22          2021  TV-PG    91 min   
4  United States 2021-09-24          1993  TV-MA   125 m

In [8]:
# standardizing type
df["type"] = df["type"].str.title()
df["type"] = df["type"].replace({
    "Tv Show": "TV Show",
    "Tv Show ": "TV Show",
    "movie": "Movie",
    "tv show": "TV Show"
})
print(df["type"].value_counts())

type
Movie      6126
TV Show    2664
Name: count, dtype: int64


In [9]:
# standardizing rating
df["rating"] = df["rating"].str.upper()

df["rating"] = df["rating"].replace({
    "NOT RATED": "NR",
    "N/R": "NR",
    "TV MA": "TV-MA",
    "TV PG": "TV-PG",
    "TV G": "TV-G",
    "TV 14": "TV-14",
    "TV Y": "TV-Y",
    "TV Y7": "TV-Y7",
    "TV Y7 FV": "TV-Y7-FV",
    "UR ": "UR"
})
print(df["rating"].value_counts())

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [10]:
df["country"] = df["country"].astype(str).str.strip()
df["country"] = df["country"].replace({
    "Not Given": "Unknown Country",
    "Unknown": "Unknown Country",
    "not given": "Unknown Country"
})
df["country"] = df["country"].str.title()

df["country"] = df["country"].str.replace(" / ", ", ", regex=False)
df["country"] = df["country"].str.replace("/", ", ", regex=False)

print(df["country"].value_counts())

country
United States      3240
India              1057
United Kingdom      638
Pakistan            421
Unknown Country     287
                   ... 
Luxembourg            1
Senegal               1
Belarus               1
Puerto Rico           1
Cyprus                1
Name: count, Length: 86, dtype: int64


In [11]:
# check to confirm if cleaning worked
print(df.head())
print(df.isna().sum())
print(df["type"].value_counts())
print(df["rating"].value_counts())
print(df["country"].value_counts())

  show_id     type                             title         director  \
0      s1    Movie              Dick Johnson Is Dead  Kirsten Johnson   
1      s3  TV Show                         Ganglands  Julien Leclercq   
2      s6  TV Show                     Midnight Mass    Mike Flanagan   
3     s14    Movie  Confessions of an Invisible Girl    Bruno Garotti   
4      s8    Movie                           Sankofa     Haile Gerima   

         country date_added  release_year rating  duration  \
0  United States 2021-09-25          2020  PG-13    90 min   
1         France 2021-09-24          2021  TV-MA  1 Season   
2  United States 2021-09-24          2021  TV-MA  1 Season   
3         Brazil 2021-09-22          2021  TV-PG    91 min   
4  United States 2021-09-24          1993  TV-MA   125 min   

                                           listed_in  
0                                      Documentaries  
1  Crime TV Shows, International TV Shows, TV Act...  
2                 TV Dr

In [12]:
# exporting cleaned dataset
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "netflix_cleaned_dataset.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df):,} rows to {output_path}")

Saved 8,790 rows to ../data/processed/netflix_cleaned_dataset.csv


In [13]:
# verifying the export
check_df = pd.read_csv(output_path)
print("Exported shape:", check_df.shape)
print("Columns:", list(check_df.columns))
print("Duplicate rows:", check_df.duplicated().sum())

Exported shape: (8790, 10)
Columns: ['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in']
Duplicate rows: 0
